<a href="https://colab.research.google.com/github/GelsonRibeiroJr/alura-agent-rag/blob/main/1%C2%BA_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Importando Bibliotecas*

In [11]:
%pip install -qU pypdf
%pip install -U langchain
%pip install -U langchain-community
%pip install -U langchain-groq
%pip install langchain-huggingface
%pip install langgraph

Chave de API - **GROQ & NASA**

In [12]:
from google.colab import userdata
import os

def carregar_chave(nome_secret):
    valor = userdata.get(nome_secret)
    if not valor:
        raise ValueError(f"❌ Secret '{nome_secret}' não encontrada ou vazia. Configure em Colab > Secrets.")
    os.environ[nome_secret] = valor
    return valor

# Carrega a chave da Groq
groq_api_key = carregar_chave('GROQ_API_KEY')

# Carrega a chave da NASA
nasa_api_key = carregar_chave('NASA_API_KEY')

# Carrega o token do Hugging Face
hf_token = carregar_chave('HF_TOKEN')

print("✅ Chaves de API carregadas com sucesso!")

✅ Chaves de API carregadas com sucesso!


Cloando o repositorio para o ambiente Colab

In [13]:
!git clone https://github.com/GelsonRibeiroJr/alura-agent-rag.git

fatal: destination path 'alura-agent-rag' already exists and is not an empty directory.


In [14]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

# Caminho apontando para a pasta baixada do GitHub
path_data = './alura-agent-rag/data'

# Configura o leitor para processar todos os PDFs
loader = DirectoryLoader(
    path_data,
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=True
)

# Carrega todos os documentos
documents = loader.load()

print(f"Sucesso! Total de páginas processadas: {len(documents)}")

100%|██████████| 29/29 [00:34<00:00,  1.20s/it]

Sucesso! Total de páginas processadas: 400


In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configura o fatiador de texto
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,   # Tamanho aproximado de cada pedaço (caracteres)
    chunk_overlap=150  # Quantidade de caracteres sobrepostos entre pedaços vizinhos
)

# Aplica a divisão em todas as 400 páginas
chunks = text_splitter.split_documents(documents)

print(f"Base fatiada com sucesso! Total de chunks gerados: {len(chunks)}")
print("\n--- Exemplo do primeiro Chunk gerado ---")
print(chunks[0].page_content[:300]) # Mostra os primeiros 300 caracteres do 1º pedaço

Base fatiada com sucesso! Total de chunks gerados: 890

--- Exemplo do primeiro Chunk gerado ---
National Aeronautics and Space Administration
Geology Training for Artemis Missions
National Academies Panel on Lunar and 
Planetary Sciences for Key Non-Polar 
Destinations Across the Moon to Address 
Decadal-level Science Objectives with 
Human Explorers 
Cynthia Evans, Ph.D. 
Artemis Geology Trai


In [16]:
# 1. Instala a biblioteca de embeddings da HuggingFace e o banco vetorial FAISS
%pip install -q sentence-transformers faiss-cpu

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Carregando o modelo de Embeddings...")
# Usamos um modelo multilíngue super eficiente e leve
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

print(f"Gerando os vetores no banco de dados (FAISS)... Isso leva cerca de 30 a 60 segundos.")
vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

# Salva o índice localmente para não precisar reprocessar tudo se reiniciar o ambiente
vectorstore.save_local("faiss_index")

print(f"✅ Banco Vetorial criado com sucesso! Todos os {len(chunks)} chunks foram indexados.")

Carregando o modelo de Embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Gerando os vetores no banco de dados (FAISS)... Isso leva cerca de 30 a 60 segundos.
✅ Banco Vetorial criado com sucesso! Todos os 890 chunks foram indexados.


In [17]:
# Teste de busca por similaridade semântica
pergunta = "Qual é o objetivo do programa Human Landing System no projeto Artemis?"

# Busca os 3 pedaços de texto mais parecidos no banco
resultados = vectorstore.similarity_search(pergunta, k=3)

print("--- Trechos mais relevantes encontrados pelo Banco Vetorial ---")
for i, doc in enumerate(resultados):
    print(f"\nResultado {i+1}:")
    print(doc.page_content[:400]) # Exibe os 400 primeiros caracteres do resultado

--- Trechos mais relevantes encontrados pelo Banco Vetorial ---

Resultado 1:
AAS 23-057
AN OVERVIEW OF THE ARTEMIS I NAVIGATION PERFORMANCE
Greg Holt*, Chris D’Souza †, and Michael Wasinger ‡
The goal of NASA’s Artemis Program is to explore the Moon and beyond. The
Artemis I Mission which flew in late 2022 was the uncrewed test flight whose goal
was to exercise the entire navigation system in an extended duration flight and
evaluate its performance over the entire mission,

Resultado 2:
Artemis program will land the first woman and next man on the surface of the Moon and 
establish, together with international and commercial partners, the sustainable human exploration 
of the solar system; 
 
CONSIDERING the necessity of greater coordination and cooperation between and among 
established and emerging actors in space; 
 
RECOGNIZING the global benefits of space exploration and com

Resultado 3:
11
Chapter 1: Setting Humanity on a Sustainable Course  
to the Moon
The Artemis program bui

Importando **ChatGroq**

In [18]:
import os
from langchain_groq import ChatGroq

# Inicializa o modelo LLM da Groq
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2, # Baixa temperatura para manter fidelidade aos documentos
    api_key=os.environ.get("GROQ_API_KEY")
)

# Teste rápido direto do modelo (sem RAG)
resposta_teste = llm.invoke("Diga 'Modelo Groq conectado com sucesso!' em português.")
print(resposta_teste.content)

"Modelo Groq conectado com sucesso!"


Configurando a Cadeia RAG

In [33]:
import requests
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# --- FUNÇÃO DE CONSULTA À API DA NASA (FALLBACK) ---
def buscar_api_nasa(inputs) -> str:
    """Busca informações na API da NASA caso a pergunta seja geral."""
    # Trata caso receba string ou dicionário sem quebrar
    if isinstance(inputs, dict):
        query = inputs.get("question", "")
    else:
        query = str(inputs)

    try:
        url = f"https://images-api.nasa.gov/search?q={query}&media_type=image"
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            items = response.json().get("collection", {}).get("items", [])
            if items:
                desc = items[0]["data"][0].get("description", "")
                return desc[:500]  # Limita aos 500 primeiros caracteres
    except Exception:
        pass
    return "Nenhum dado retornado da API."

# --- HELPER DE FORMATAÇÃO COM METADADOS DE PÁGINA ---
def format_docs_com_metadados(docs):
    """Formata os trechos encontrados incluindo o nome do PDF e a página."""
    texto_formatado = []
    for doc in docs:
        fonte = doc.metadata.get('source', 'Documento NASA').split('/')[-1]
        pagina = doc.metadata.get('page', 0) + 1
        texto_formatado.append(f"[Documento: '{fonte}', Página {pagina}]\n{doc.page_content}")
    return "\n\n---\n\n".join(texto_formatado)

# --- PROMPT DIRETO E SEM RODEIOS ---
template = """Você é um assistente técnico especialista da NASA.
Responda à pergunta do usuário de forma DIRETA, OBJETIVA e em Português do Brasil.
Não faça justificativas, introduções ou explicações sobre a busca. Vá direto à resposta e seja fiel a suas fontes.

REGRA DE FONTE:
- Se a resposta foi encontrada na "Base Interna (PDFs)", coloque na última linha: "Fonte: Documento 'NOME_DO_ARQUIVO.pdf', Página X"
- Se a resposta veio da chave de API "NASA_API_KEY", coloque na última linha: "Fonte: API da NASA"

Base Interna (PDFs):
{context}

Dados da API da NASA:
{api_context}

Pergunta do usuário:
{question}

Resposta direta:"""

prompt = ChatPromptTemplate.from_template(template)

# Configura o recuperador do FAISS
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# --- CADEIA RAG HÍBRIDA ---
rag_chain = (
    {
        "context": retriever | format_docs_com_metadados,
        "api_context": buscar_api_nasa,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ Pipeline RAG Híbrido ajustado e pronto!")

✅ Pipeline RAG Híbrido ajustado e pronto!


Teste do RAG em Funcionamento

In [37]:
# Pergunta dentro do escopo dos docs em PDF para testar a citação da página!
pergunta_doc = "Qual foi o objetivo da missão Artemis I?"
print("=== TESTE 1: BUSCA NOS DOCUMENTOS ===")
print(rag_chain.invoke(pergunta_doc))

=== TESTE 1: BUSCA NOS DOCUMENTOS ===
O objetivo da missão Artemis I foi exercitar o sistema de navegação inteiro em uma viagem de longa duração e avaliar seu desempenho ao longo da missão, desde a pré-lançamento até a pós-pousa.

Fonte: Documento 'AAS-23-057_Artemis_1_Navigation_HoltDsouzaWasinger_FINALfinal.pdf', Página 1


In [36]:
# Pergunta sobre algo fora do escopo dos PDFs da missão Artemis
pergunta_api = "O que é o telescópio James Webb?"
print("\n=== TESTE 2: BUSCA FORA DOS DOCS (USANDO API) ===")
print(rag_chain.invoke(pergunta_api))


=== TESTE 2: BUSCA FORA DOS DOCS (USANDO API) ===
O telescópio James Webb é um telescópio espacial que foi lançado em 25 de dezembro de 2021 e é uma missão conjunta da NASA, da Agência Espacial Europeia (ESA), da Agência Espacial Canadense (CSA) e da Agência Espacial Alemã (DLR). Ele é o sucessor do telescópio Hubble e é projetado para observar o universo em longas ondas de luz, desde as primeiras estrelas e galáxias até os objetos mais distantes do universo.

Fonte: Documento 'jwst.pdf', Página 1


Evoluindo para Multi-Tool Agents

In [38]:
from langchain_core.tools import tool
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS

# --- 1. FERRAMENTA PRINCIPAL: PROGRAMA ARTEMIS E MISSÕES LUNARES (SEU FAISS) ---
@tool
def pega_contexto_artemis_lunar(query: str) -> str:
    """Pega o contexto sobre o programa espacial Artemis, naves, rotas e pousadores na Lua baseado nos PDFs oficiais."""
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    resultado = retriever.invoke(query)

    texto_formatado = []
    for doc in resultado:
        fonte = doc.metadata.get('source', 'NASA').split('/')[-1]
        pagina = doc.metadata.get('page', 0) + 1
        texto_formatado.append(f"[Documento: '{fonte}', Página {pagina}]\n{doc.page_content}")

    return "\n\n---\n\n".join(texto_formatado)

In [39]:
# --- 2. FERRAMENTA SECUNDÁRIA: CULTIVO E AGRICULTURA ESPACIAL ---
# Usamos a URL do curso para carregar os dados de agricultura, mas registramos a Tool focado em botânica espacial
def carrega_pdf_externo(url: str):
    loader = PyPDFLoader(url)
    pages = loader.load()
    return FAISS.from_documents(pages, embeddings)

print("Carregando bases complementares da NASA...")
vector_store_agri = carrega_pdf_externo('https://raw.githubusercontent.com/allanspadini/curso-flash-rag/main/agriculture.pdf')

@tool
def pega_contexto_agricultura_espacial(query: str) -> str:
    """Pega informações sobre agricultura, solos, biologia vegetal e produção de alimentos para suporte de vida espacial."""
    retriever = vector_store_agri.as_retriever(search_kwargs={"k": 2})
    resultado = retriever.invoke(query)
    return "\n\n".join([f"[Fonte: Pesquisa Agrícola/Botânica NASA]\n{doc.page_content}" for doc in resultado])


# --- 3. LISTA DE FERRAMENTAS DO AGENTE ---
tools = [pega_contexto_artemis_lunar, pega_contexto_agricultura_espacial]

print("✅ Ferramentas do Agente NASA configuradas com sucesso!")

Carregando bases complementares da NASA...
✅ Ferramentas do Agente NASA configuradas com sucesso!


Teste de Validação da Lista Tools

In [41]:
# Teste de invocação direta da ferramenta de botânica/agricultura espacial
resultado_agri = pega_contexto_agricultura_espacial.invoke("Como é realizada a pesquisa de cultivo de plantas?")

print("=== CONTEXTO RESGATADO PELA FERRAMENTA ESPACIAL ===")
print(resultado_agri[:400])

=== CONTEXTO RESGATADO PELA FERRAMENTA ESPACIAL ===
[Fonte: Pesquisa Agrícola/Botânica NASA]
Analysis of Crop Production Dataset using R Tool 
108 
Published By: 
Blue Eyes Intelligence Engineering  
and Sciences Publication (BEIESP)  
© Copyright: All rights reserved. 
 
Retrieval Number: A11001291S419/2019©BEIESP 
DOI:10.35940/ijeat.A1100.1291S419 
Journal Website: www.ijeat.org 
 
The dataset is analyzed for finding the crops that are 
produced 
